In [ ]:
```xml
<VSCode.Cell language="markdown">
# Ch.7 — Networking & Load Balancing

**Running example:** Nginx reverse proxy + 3 Flask replicas

This notebook demonstrates:
- Deploying multiple backend replicas
- Configuring Nginx as reverse proxy
- Load distribution (round-robin, least-conn, ip-hash)
- Health checks and failover
- Sticky sessions, SSL termination, rate limiting
</VSCode.Cell>
<VSCode.Cell language="python">
# Cell 1: Deploy Multiple Flask Replicas
# Create a simple Flask app with health check and unique backend ID

import os
import subprocess

# Create Flask app
app_code = '''from flask import Flask, jsonify
import os
import time

app = Flask(__name__)
BACKEND_ID = os.getenv("BACKEND_ID", "unknown")

@app.route("/health")
def health():
    """Health check endpoint for Nginx"""
    return jsonify({"status": "healthy", "backend": BACKEND_ID}), 200

@app.route("/api/payment")
def payment():
    """Example API endpoint"""
    time.sleep(0.1)  # Simulate processing
    return jsonify({
        "message": "Payment processed",
        "backend": BACKEND_ID,
        "timestamp": time.time()
    }), 200

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)
'''

# Write app.py
with open("app.py", "w") as f:
    f.write(app_code)

# Create Dockerfile
dockerfile = '''FROM python:3.11-slim
WORKDIR /app
RUN pip install flask
COPY app.py .
CMD ["python", "app.py"]
'''

with open("Dockerfile", "w") as f:
    f.write(dockerfile)

# Build image
print("Building Flask app image...")
subprocess.run(["docker", "build", "-t", "flask-backend:latest", "."], check=True)
print("✓ Flask backend image built")
</VSCode.Cell>
<VSCode.Cell language="python">
# Cell 2: Write nginx.conf (Reverse Proxy Configuration)

nginx_conf = '''events {
    worker_connections 1024;
}

http {
    # Define upstream backend pool
    upstream backend {
        # Round-robin (default)
        server backend1:5000 max_fails=2 fail_timeout=30s;
        server backend2:5000 max_fails=2 fail_timeout=30s;
        server backend3:5000 max_fails=2 fail_timeout=30s;
    }

    # Access log format with backend info
    log_format upstreamlog '$remote_addr - $remote_user [$time_local] '
                           '"$request" $status $body_bytes_sent '
                           '"$http_referer" "$http_user_agent" '
                           'upstream: $upstream_addr';

    server {
        listen 80;
        
        access_log /var/log/nginx/access.log upstreamlog;
        
        location / {
            proxy_pass http://backend;
            proxy_set_header Host $host;
            proxy_set_header X-Real-IP $remote_addr;
            proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
            
            # Timeouts
            proxy_connect_timeout 5s;
            proxy_send_timeout 10s;
            proxy_read_timeout 10s;
        }
    }
}
'''

with open("nginx.conf", "w") as f:
    f.write(nginx_conf)

print("✓ nginx.conf created")
print("\nConfiguration highlights:")
print("- Upstream pool: backend1, backend2, backend3")
print("- Algorithm: round-robin (default)")
print("- Health check: passive (max_fails=2, fail_timeout=30s)")
print("- Timeouts: 5s connect, 10s send/read")
</VSCode.Cell>
<VSCode.Cell language="python">
# Cell 3: Deploy Nginx Container with Docker Compose

docker_compose = '''version: '3.8'

services:
  nginx:
    image: nginx:latest
    ports:
      - "8080:80"
    volumes:
      - ./nginx.conf:/etc/nginx/nginx.conf:ro
    depends_on:
      - backend1
      - backend2
      - backend3
    networks:
      - app-network

  backend1:
    image: flask-backend:latest
    environment:
      - BACKEND_ID=backend1
    networks:
      - app-network

  backend2:
    image: flask-backend:latest
    environment:
      - BACKEND_ID=backend2
    networks:
      - app-network

  backend3:
    image: flask-backend:latest
    environment:
      - BACKEND_ID=backend3
    networks:
      - app-network

networks:
  app-network:
    driver: bridge
'''

with open("docker-compose.yml", "w") as f:
    f.write(docker_compose)

# Start the stack
print("Starting Nginx + 3 Flask replicas...")
subprocess.run(["docker", "compose", "up", "-d"], check=True)

# Wait for services to be ready
import time
time.sleep(5)

# Check status
result = subprocess.run(["docker", "compose", "ps"], capture_output=True, text=True)
print("\n✓ Stack running:")
print(result.stdout)

print("\nAccess points:")
print("- Nginx: http://localhost:8080")
print("- Backend1: backend1:5000 (internal)")
print("- Backend2: backend2:5000 (internal)")
print("- Backend3: backend3:5000 (internal)")
</VSCode.Cell>
<VSCode.Cell language="python">
# Cell 4: Send Requests, Observe Load Distribution

import requests
import json
from collections import Counter

# Send 30 requests to observe load distribution
responses = []
print("Sending 30 requests to http://localhost:8080/api/payment...")

for i in range(30):
    try:
        resp = requests.get("http://localhost:8080/api/payment", timeout=2)
        data = resp.json()
        responses.append(data["backend"])
        print(f"Request {i+1}: {data['backend']}", end="\r")
    except Exception as e:
        print(f"Request {i+1} failed: {e}")

print("\n\n✓ Load distribution (round-robin):")
backend_counts = Counter(responses)
for backend, count in sorted(backend_counts.items()):
    bar = "█" * count
    print(f"{backend}: {count:2d} requests {bar}")

print(f"\nTotal: {sum(backend_counts.values())} successful requests")
print("\nExpected: ~10 requests per backend (even distribution)")

# Verify round-robin pattern
if len(backend_counts) == 3:
    print("✓ All 3 backends received traffic")
else:
    print("⚠ Warning: Some backends did not receive traffic")
</VSCode.Cell>
<VSCode.Cell language="python">
# Cell 5: Simulate Backend Failure (Stop One Container)

print("Simulating backend2 failure...")
subprocess.run(["docker", "compose", "stop", "backend2"], check=True)
print("✓ backend2 stopped")

# Wait for Nginx to detect failure
import time
time.sleep(2)

# Send 20 requests
responses = []
print("\nSending 20 requests after backend2 failure...")

for i in range(20):
    try:
        resp = requests.get("http://localhost:8080/api/payment", timeout=2)
        data = resp.json()
        responses.append(data["backend"])
        print(f"Request {i+1}: {data['backend']}", end="\r")
    except Exception as e:
        responses.append("FAILED")
        print(f"Request {i+1}: FAILED ({e})")

print("\n\n✓ Load distribution after backend2 failure:")
backend_counts = Counter(responses)
for backend, count in sorted(backend_counts.items()):
    bar = "█" * count
    print(f"{backend}: {count:2d} requests {bar}")

print("\nExpected behavior:")
print("- backend2: 0 requests (down)")
print("- backend1 & backend3: ~10 requests each (failover)")
print("- Some initial failures before Nginx detects backend2 is down")

# Restart backend2
print("\nRestarting backend2...")
subprocess.run(["docker", "compose", "start", "backend2"], check=True)
time.sleep(2)
print("✓ backend2 restarted")
</VSCode.Cell>
<VSCode.Cell language="python">
# Cell 6: Health Checks (Passive vs Active)

print("=== Passive Health Checks (Current Configuration) ===\n")
print("nginx.conf setting: max_fails=2 fail_timeout=30s")
print("\nBehavior:")
print("- Nginx marks backend down after 2 consecutive failures")
print("- Retries backend after 30 seconds")
print("- No extra probe traffic (only real request failures)")

print("\n=== Active Health Checks (Nginx Plus / Third-Party Module) ===\n")
print("Configuration example:")
active_health_check = '''http {
    upstream backend {
        server backend1:5000;
        server backend2:5000;
        server backend3:5000;
        
        # Active health check (Nginx Plus)
        health_check interval=5s fails=2 passes=2 uri=/health;
    }
}
'''
print(active_health_check)

print("Behavior:")
print("- Nginx probes /health endpoint every 5 seconds")
print("- Marks backend down after 2 failed probes")
print("- Marks backend up after 2 successful probes")
print("- Faster failure detection (5-10s vs 30s+ for passive)")

# Test health endpoint
print("\nTesting health endpoints:")
for backend in ["backend1", "backend2", "backend3"]:
    try:
        # Use Docker network to access backends directly (for demo)
        result = subprocess.run(
            ["docker", "compose", "exec", "-T", backend, "curl", "-s", "http://localhost:5000/health"],
            capture_output=True,
            text=True,
            timeout=2
        )
        print(f"✓ {backend}/health: {result.stdout.strip()}")
    except Exception as e:
        print(f"✗ {backend}/health: Failed ({e})")
</VSCode.Cell>
<VSCode.Cell language="python">
# Cell 7: Sticky Sessions (Session Affinity with ip_hash)

print("=== Testing Sticky Sessions with ip_hash ===\n")

# Create new nginx.conf with ip_hash
nginx_conf_sticky = '''events {
    worker_connections 1024;
}

http {
    upstream backend {
        ip_hash;  # Enable session affinity
        server backend1:5000;
        server backend2:5000;
        server backend3:5000;
    }

    log_format upstreamlog '$remote_addr - $remote_user [$time_local] '
                           '"$request" $status $body_bytes_sent '
                           'upstream: $upstream_addr';

    server {
        listen 80;
        access_log /var/log/nginx/access.log upstreamlog;
        
        location / {
            proxy_pass http://backend;
            proxy_set_header Host $host;
            proxy_set_header X-Real-IP $remote_addr;
            proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        }
    }
}
'''

with open("nginx.conf", "w") as f:
    f.write(nginx_conf_sticky)

# Reload Nginx config
print("Reloading Nginx with ip_hash configuration...")
subprocess.run(["docker", "compose", "exec", "nginx", "nginx", "-s", "reload"], check=True)
time.sleep(1)

# Send 20 requests from same client (should all go to same backend)
responses = []
print("\nSending 20 requests with ip_hash...")

for i in range(20):
    try:
        resp = requests.get("http://localhost:8080/api/payment", timeout=2)
        data = resp.json()
        responses.append(data["backend"])
    except Exception as e:
        print(f"Request {i+1} failed: {e}")

print("\n✓ Load distribution with ip_hash:")
backend_counts = Counter(responses)
for backend, count in sorted(backend_counts.items()):
    bar = "█" * count
    print(f"{backend}: {count:2d} requests {bar}")

print("\nExpected: All 20 requests to same backend (sticky session)")
unique_backends = len(backend_counts)
if unique_backends == 1:
    print("✓ Session affinity working (all requests to one backend)")
else:
    print(f"⚠ Warning: Requests distributed across {unique_backends} backends")
    print("  (ip_hash may vary based on Docker network setup)")
</VSCode.Cell>
<VSCode.Cell language="python">
# Cell 8: SSL Termination at Proxy

print("=== SSL Termination Configuration ===\n")

# Generate self-signed certificate (for demo purposes)
print("Generating self-signed SSL certificate...")
subprocess.run([
    "openssl", "req", "-x509", "-nodes", "-days", "365",
    "-newkey", "rsa:2048",
    "-keyout", "nginx-selfsigned.key",
    "-out", "nginx-selfsigned.crt",
    "-subj", "/CN=localhost"
], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("✓ SSL certificate created: nginx-selfsigned.crt")
print("✓ SSL key created: nginx-selfsigned.key")

# Nginx config with SSL termination
nginx_conf_ssl = '''events {
    worker_connections 1024;
}

http {
    upstream backend {
        server backend1:5000;
        server backend2:5000;
        server backend3:5000;
    }

    # HTTP server (redirect to HTTPS)
    server {
        listen 80;
        server_name localhost;
        return 301 https://$server_name$request_uri;
    }

    # HTTPS server (SSL termination)
    server {
        listen 443 ssl;
        server_name localhost;
        
        ssl_certificate /etc/nginx/certs/nginx-selfsigned.crt;
        ssl_certificate_key /etc/nginx/certs/nginx-selfsigned.key;
        
        ssl_protocols TLSv1.2 TLSv1.3;
        ssl_ciphers HIGH:!aNULL:!MD5;
        
        location / {
            proxy_pass http://backend;  # Forward HTTP to backends
            proxy_set_header Host $host;
            proxy_set_header X-Real-IP $remote_addr;
            proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
            proxy_set_header X-Forwarded-Proto https;  # Tell backend original protocol
        }
    }
}
'''

print("\nSSL Termination Architecture:")
print("1. Client → Nginx (HTTPS, port 443)")
print("2. Nginx decrypts HTTPS → HTTP")
print("3. Nginx → Backend (HTTP, port 5000)")
print("4. Backend → Nginx (HTTP response)")
print("5. Nginx encrypts HTTP → HTTPS")
print("6. Nginx → Client (HTTPS response)")

print("\nBenefits:")
print("- Centralized certificate management (one cert for all backends)")
print("- Reduced backend CPU load (no SSL processing)")
print("- Simplified backend code (no HTTPS handling)")

print("\nNote: To enable SSL, update docker-compose.yml to mount certificates:")
print("  volumes:")
print("    - ./nginx.conf:/etc/nginx/nginx.conf:ro")
print("    - ./nginx-selfsigned.crt:/etc/nginx/certs/nginx-selfsigned.crt:ro")
print("    - ./nginx-selfsigned.key:/etc/nginx/certs/nginx-selfsigned.key:ro")
</VSCode.Cell>
<VSCode.Cell language="python">
# Cell 9: Rate Limiting (Protect Backends from Overload)

print("=== Rate Limiting Configuration ===\n")

# Nginx config with rate limiting
nginx_conf_ratelimit = '''events {
    worker_connections 1024;
}

http {
    # Rate limiting zone: 10MB memory, max 10 requests/sec per IP
    limit_req_zone $binary_remote_addr zone=one:10m rate=10r/s;
    
    upstream backend {
        server backend1:5000;
        server backend2:5000;
        server backend3:5000;
    }

    server {
        listen 80;
        
        location / {
            # Apply rate limit: 10 req/sec, burst up to 20, delay excess requests
            limit_req zone=one burst=20 nodelay;
            
            proxy_pass http://backend;
            proxy_set_header Host $host;
            proxy_set_header X-Real-IP $remote_addr;
            proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        }
    }
}
'''

with open("nginx.conf", "w") as f:
    f.write(nginx_conf_ratelimit)

# Reload Nginx
print("Reloading Nginx with rate limiting...")
subprocess.run(["docker", "compose", "exec", "nginx", "nginx", "-s", "reload"], check=True)
time.sleep(1)

# Test rate limiting by sending rapid requests
import time
print("\nTesting rate limit (sending 30 rapid requests)...")

success_count = 0
rate_limited_count = 0
start_time = time.time()

for i in range(30):
    try:
        resp = requests.get("http://localhost:8080/api/payment", timeout=2)
        if resp.status_code == 200:
            success_count += 1
            print(f"Request {i+1}: SUCCESS", end="\r")
        elif resp.status_code == 503:  # Service Unavailable (rate limited)
            rate_limited_count += 1
            print(f"Request {i+1}: RATE LIMITED (503)", end="\r")
    except Exception as e:
        print(f"Request {i+1}: ERROR ({e})")
    time.sleep(0.05)  # 50ms delay (20 req/sec)

elapsed = time.time() - start_time

print(f"\n\n✓ Results:")
print(f"- Successful: {success_count}")
print(f"- Rate limited (503): {rate_limited_count}")
print(f"- Time elapsed: {elapsed:.2f}s")
print(f"- Effective rate: {success_count/elapsed:.1f} req/sec")

print("\nConfiguration:")
print("- Rate limit: 10 req/sec per IP")
print("- Burst: 20 requests (allows brief spikes)")
print("- Mode: nodelay (reject immediately when over limit)")

print("\nProduction best practices:")
print("- Set rate limit based on backend capacity")
print("- Monitor 503 responses (users being rate limited)")
print("- Consider per-route limits (stricter on expensive endpoints)")
</VSCode.Cell>
<VSCode.Cell language="python">
# Cell 10: Logging and Access Logs Analysis

print("=== Access Logs Analysis ===\n")

# Get Nginx access logs
print("Recent Nginx access logs:")
result = subprocess.run(
    ["docker", "compose", "logs", "--tail=20", "nginx"],
    capture_output=True,
    text=True
)

# Parse logs to show upstream routing
logs = result.stdout.split("\n")
upstream_logs = [line for line in logs if "upstream:" in line]

if upstream_logs:
    print(f"\nShowing last {len(upstream_logs)} requests with upstream info:")
    for log in upstream_logs[-10:]:  # Show last 10
        # Extract relevant parts
        if "upstream:" in log:
            parts = log.split("upstream:")
            if len(parts) > 1:
                upstream = parts[1].strip().split()[0]
                print(f"  → {upstream}")
else:
    print("No upstream logs found (custom log format may not be active)")

# Get backend logs
print("\n=== Backend Logs ===")
for backend in ["backend1", "backend2", "backend3"]:
    result = subprocess.run(
        ["docker", "compose", "logs", "--tail=3", backend],
        capture_output=True,
        text=True
    )
    log_lines = [line for line in result.stdout.split("\n") if line.strip()]
    if log_lines:
        print(f"\n{backend} (last 3 requests):")
        for line in log_lines[-3:]:
            if "GET" in line or "POST" in line:
                print(f"  {line.split('|')[-1].strip() if '|' in line else line.strip()}")

print("\n\n=== Useful Log Analysis Commands ===")
print("1. Count requests per backend:")
print("   docker compose logs nginx | grep 'upstream:' | awk '{print $NF}' | sort | uniq -c")

print("\n2. Monitor live traffic:")
print("   docker compose logs -f nginx")

print("\n3. Check backend health check failures:")
print("   docker compose logs nginx | grep -i 'upstream timed out\\|502'")

print("\n4. View backend response times:")
print("   docker compose logs nginx | grep 'upstream_response_time'")

print("\n\n✓ Load balancing demonstration complete!")
print("\nCleanup:")
print("  docker compose down")
</VSCode.Cell>
```